# Day 21 · OpenStreetMap — ทะเบียน 83 แหล่งดูฟ้ามืด OSM รู้จักกี่แห่ง?

**#30DayMapChallenge 2026 · 21 พ.ย. 2569** · โจทย์:
*"The awesome data source for mapping. You can also use this day to edit OSM."*

คืนนี้ Overpass ทุก mirror ตอบ 502/504 พร้อมกัน (คนทำ Day 21 ทั่วโลกยิงพร้อมกันพอดี)
โน้ตบุ๊กนี้จึงเดินเส้นทางที่สองของการใช้ OSM: **ดาวน์โหลดไฟล์ extract ของประเทศไทยจาก Geofabrik
(330,640,775 ไบต์ · md5 `db3958a037ce717c462101b86e2f75f3`) แล้วสแกนในเครื่องด้วย `pyosmium`**
ไม่ต้องพึ่ง API เลย และได้ของแถมสองอย่าง: แท็กทั้งหมดของวัตถุ กับ geometry จริงของ way

| นิยาม | ความหมาย |
|---|---|
| จานสแกน | รัศมี 2,000 ม. รอบจุดทะเบียน — วัตถุแตะจาน = เข้าชุดข้อมูล |
| ระยะจับคู่ | ภายใน 1,200 ม. จึงนับว่า "วัตถุนี้อยู่ที่นี่" |
| ระยะประเภทรายวัน | ภายใน 300 ม. จึงนับว่ามีวัตถุประเภทที่พัก/แหล่งเที่ยวตรงตำแหน่งนั้น |
| การจับชื่อ | 4 การทดสอบ: ชื่อไทยตรงกันทุกตัวอักษร · รูปละตินของ slug ทางการตรงกับชื่อ OSM · คำหลักละตินตรงกันเกินครึ่ง · similarity ≥ 0.78 |
| เพดานระยะตามความแรงหลักฐาน | ชื่อเต็ม 1,200 ม. · คล้ายตามตัวอักษร 600 ม. · คำหลัก 400 ม. (คู่ที่เกินถูกตัดทิ้งและบันทึกไว้ใน `false_positives_rejected`) |
| ในผืน protection | จุดทะเบียนอยู่ *ภายใน* polygon ที่ OSM วาดเป็นพื้นที่คุ้มครอง (STRtree, ไม่ขึ้นกับรัศมีจาน) |

**รัน 3 ไฟล์ตามลำดับ** (ไฟล์แรกกับไฟล์สองใช้เวลารวม ~10 นาทีบนเครื่อง เพราะต้องสแกน
node 44,124,360 ตัวสามรอบ)

1. `tools/collect_day21_pbf.py` → `cache/day21/{poi_nodes,ways,relations}.json.gz` + `way_geom.json.gz`
2. `tools/collect_day21_polygons.py` → `cache/day21/{polygons.json,poly_draw.json,prot_ways.json.gz}`
   (ขั้นที่ปิดบั๊ก "ขอบอุทยานใหญ่กว่าจานสแกน")
3. `tools/day21_osm.py` → ภาพ 3 แผ่น + csv 3 ไฟล์ + geojson 1 ไฟล์ + `day21_stats.json`

**สคริปต์ทั้งหมดอ่านอย่างเดียว** — ไม่ล็อกอิน ไม่ส่ง changeset เข้า OpenStreetMap
รายการที่ควรเพิ่ม/แก้ 48 จุดออกเป็น `day21_edit_queue.csv` ให้คนตรวจก่อนเสมอ
เพราะการยกข้อมูลจากฐานอื่นลง OSM ทั้งชุดถือเป็น *import* ซึ่งมีขั้นตอนของมัน
(ดู Import/Guidelines ใน OSM wiki)

## ถ้าจะรันบน Colab (ใช้เวลา ~12 นาที + ดิสก์ ~1 GB)

โน้ตบุ๊กนี้ถูกสร้างจาก `tools/day21_osm.py` ซึ่งอ่านผลสแกนจาก `cache/day21/`
บน Colab ต้องสแกนเองใหม่ทั้งหมด (repo มีโค้ดสแกนครบสามไฟล์แล้ว) รัน cell นี้ก่อน:

```python
import os, subprocess, sys
if not os.path.exists("/content/geo-notes"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/thammarat-ai/geo-notes"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","h3==4.5.0","pyosmium","geopandas","pyproj"], check=True)
os.makedirs("/content/cache/day21", exist_ok=True)
if not os.path.exists("/content/cache/day21/thailand.osm.pbf"):
    subprocess.run(["curl","-L","-o","/content/cache/day21/thailand.osm.pbf",
                    "https://download.geofabrik.de/asia/thailand/thailand-latest.osm.pbf"], check=True)
```

แล้วเปลี่ยนค่า `WORK` ในสามสคริปต์เป็น `/content` (สคริปต์อื่นในซีรีส์มีตัวช่วยนี้แล้ว —
วันนี้ยัง hardcode path ของเครื่องผู้เขียนไว้ เพราะสคริปต์ต้องรันเรียงลำดับ
และ pass C ของไฟล์สแกน polygon ใช้เวลาหลายนาทีต่อครั้ง)

**คำเตือนเรื่องเวลา**: pass A/B/C สแกน node 44 ล้านตัวสามรอบ; ถ้าไม่รีบ
แนะนำให้รันเฉพาะ `tools/day21_osm.py` บนผลสแกนที่เรา publish ไว้ใน
`posts/day21-openstreetmap/day21_osm_named.geojson` + `day21_sites_osm.csv` แทน

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 21 · OpenStreetMap — "The awesome data source for mapping.
You can also use this day to edit OSM."

โจทย์ (30daymapchallenge.com หัวข้อ Day 21 'OpenStreetMap' อ่านสด 19:12 น. วันที่ 21 พ.ย. 2569)
ชวนใช้ OSM เป็นแหล่งข้อมูล และบอกว่า "จะแก้ OSM วันนี้ก็ได้" ผมจึงถามคำถามที่ตอบได้ด้วย
แหล่งข้อมูลจริงสองชิ้นที่อยู่ในมือ:

    ทะเบียนแหล่งดูฟ้ามืดของไทย (ราชการรับรอง) มี 83 แห่ง
    แล้ว OpenStreetMap "รู้จัก" กี่แห่งในนั้น และรู้จักแค่ไหน?

เส้นทางข้อมูล (ต่างจาก Day 16 โดยตั้งใจ)
  * Day 16 ยิง Overpass ขอแค่ id + user + timestamp ของ way (คำถามตอนนั้นคือ "ใครเขียน")
  * คืนนี้ Overpass คิวเต็มทุก mirror (502/504 ติดกัน 20+ นาที) จึงย้ายไปเส้นทางที่คนที่ใช้
    OSM ทั้งประเทศใช้จริง: ดาวน์โหลด Geofabrik Thailand extract (331 MB) แล้วสแกนในเครื่อง
  * ของแถมสองอย่าง: ได้ *แท็กทั้งหมด* และ *geometry จริงของ way* → วัดระยะถึง "ขอบวัตถุ"
    ไม่ใช่จุดศูนย์กลางของ way แบบ Day 16

นิยามในโพสต์ = นิยามในโค้ดนี้เท่านั้น
  R_DISC  = 2,000 ม.  จานที่สแกน (วัตถุแตะจาน = เข้าชุดข้อมูล)
  R_MATCH = 1,200 ม.  ระยะที่ถือว่า "วัตถุนี้อยู่ที่นี่" สำหรับ node/way
  R_NEAR  =   300 ม.  ระยะที่ถือว่า "วัตถุประเภทเดียวกันอยู่ตรงตำแหน่งนั้นจริง"
  หกระดับ = 1 ชื่อตรงกัน · 2 ชื่อใกล้เคียง · 2r ชื่อตรงใน relation (ขอบเขต) ·
            3 มีวัตถุประเภทเดียวกันแต่คนละชื่อ · 4 มีอย่างอื่นแต่ไม่ใช่ที่นี่ ·
            5 ไม่มีวัตถุเลยในระยะจับคู่

ไม่มีการแก้ไข OSM: อ่านไฟล์อย่างเดียว ไม่ล็อกอิน ไม่ส่ง changeset — "คิอแก้" เป็นไฟล์ให้คนตรวจ
"""
import os, sys, json, time, math, re, gzip, unicodedata, difflib, collections
from collections import Counter
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from shapely.validation import make_valid
from pyproj import CRS, Transformer
from scipy.stats import spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
FONTS = os.path.join(WORK, "cache", "day06")
OSMC = os.path.join(WORK, "cache", "day21")
OUT = os.path.join(WORK, "out", "day21")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(FONTS)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(FONTS, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day21_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

sys.path.insert(0, os.path.join(WORK, "tools"))
from guards import guard_bbox, guard_crs, guard_variance, guard_labels, lint_thai

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL, PURPLE = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf", "#c084fc"
R_DISC, R_MATCH, R_NEAR = 2000, 1200, 300
# เพดานระยะแยกตามหลักฐานของชื่อ (ตั้งหลังตรวจคู่ชื่อทุกคู่ด้วยตา — ดู scratch/day21_review.txt)
# ชื่อเต็มตรงกัน = เชื่อได้แม้จุดทะเบียนอยู่คนละฝั่งถนน / คล้ายตามตัวอักษร = ต้องใกล้กว่า 600 ม.
# คำหลักละตินตรงกัน = หลักฐานอ่อนสุด ต้องใกล้กว่า 400 ม. จึงนับ (กัน "Mae Rim ..." ชนกับ
# "Mae Rim Highway Office" คนละสถานที่ 1,041 ม.)
OK_FULL, OK_SIM, OK_TOK = 1200, 600, 400
STAY = {"camp_site", "chalet", "guest_house", "hotel", "motel", "resort", "bed_and_breakfast",
        "apartment", "wilderness_hut", "alpine_hut", "attraction", "viewpoint", "picnic_site",
        "national_park", "park", "theme_park", "zoo", "aquarium", "museum", "information",
        "artwork"}
GENERIC = {"national", "park", "the", "of", "and", "forest", "waterfall", "resort", "camp",
           "campsite", "village", "house", "hotel", "science", "technology", "memorial",
           "king", "mongkut", "beach", "farm", "guesthouse"}
KEEP_GEOM = ("name", "tourism", "amenity", "leisure", "landuse", "historic", "highway",
             "place", "shop", "natural", "waterway", "boundary", "sport", "man_made",
             "craft", "office", "railway", "aeroway")

## ตอน 1) ทะเบียน + ผลสแกนจาก PBF

In [ ]:
# ================================================== 1. ทะเบียน + ผลสแกนจาก PBF
def nk(s):
    """คีย์ไทย: NFKC + lowercase + ตัดช่อง/วรรคตอน/สัญลักษณ์ออก เทียบตัวอักษรจริง"""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFC", s).lower()
    return "".join(c for c in s if unicodedata.category(c)[0] not in ("Z", "P", "S"))

def la(s):
    """คีย์ละติน: a-z0-9 เท่านั้น รูปสะกดภาษาอังกฤษ/romanization มาชนกันตรงนี้"""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c) and c.isascii() and c.isalnum())

def latin_words(s):
    if not isinstance(s, str):
        return []
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return [w for w in re.split(r"[^a-z0-9]+", s) if len(w) >= 3]

D = pd.read_csv(os.path.join(REPO, "data", "darksky_thailand.csv"))
D["name"] = D["name"].astype(str).str.strip()
D["slug"] = D["detail_url"].astype(str).str.rstrip("/").str.split("/").str[-1]
D["key_th"] = D["name"].map(nk)
D["key_lat"] = D["slug"].map(la)
D["slug_tok"] = D["slug"].map(lambda s: {w for w in latin_words(s) if w not in GENERIC})
assert len(D) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(D)}"
P = np.array([[float(r.longitude), float(r.latitude)] for _, r in D.iterrows()])
guard_bbox(P, 96.5, 107.6, 4.3, 21.7, "darksky-points")

meta = json.load(open(os.path.join(OSMC, "scan_meta.json"), encoding="utf-8"))
assert meta["radius_m"] == R_DISC, "แคชสแกนด้วยรัศมีอื่น ต้องสแกนใหม่"
log(f"แหล่งข้อมูล: {meta['source']} · md5 {meta['md5'][:12]}… · อ่าน {meta['read_utc']} · "
    f"node ทั้งไฟล์ {meta['counts']['file_nodes']:,} way {meta['counts']['file_ways']:,}")

def gz(nm):
    with gzip.open(os.path.join(OSMC, nm), "rb") as f:
        return json.loads(f.read().decode())

nodes, ways, rels, wgeom = (gz("poi_nodes.json.gz"), gz("ways.json.gz"),
                            gz("relations.json.gz"), gz("way_geom.json.gz"))
poly = json.load(open(os.path.join(OSMC, "polygons.json"), encoding="utf-8"))
# กี่ relation ที่ตัวกรอง "มาตรฐานสากล" (boundary=protected_area) มองข้ามไป?
_pcd = json.loads(gzip.open(os.path.join(OSMC, "prot_ways.json.gz"), "rb").read().decode())
_broad = [v[0] for v in _pcd["rel"].values()]
strict = int(sum(1 for t in _broad if t.get("boundary") == "protected_area"))
missed = {k: int(sum(1 for t in _broad if t.get(k) and t.get("boundary") != "protected_area"))
          for k in ("boundary", "leisure", "tourism", "landuse", "natural")}
missed["boundary_national_park"] = int(sum(1 for t in _broad
                                           if t.get("boundary") == "national_park"))
KY = next((r for r in poly["records"] if r.get("contains") and
           "เขาใหญ่" in str(r.get("name", ""))), None)
log(f"relation พื้นที่คุ้มครอง {len(_broad)} · ที่ตัวกรองมาตรฐานเห็น {strict} · "
    f"ที่ตัวกรองนั้นมองข้าม {len(_broad) - strict}"
    + (f" · เคสที่ตรวจ: relation {KY['id']} = {KY['name']}" if KY else ""))
poly_draw = json.load(open(os.path.join(OSMC, "poly_draw.json"), encoding="utf-8"))
CONT = collections.defaultdict(list)     # จุดทะเบียน -> ผืนที่ OSM วาดห่ออยู่
EDGE = {}                                # จุดทะเบียน -> ระยะไปขอบพื้นที่คุ้มครองใกล้สุด
for r in poly["records"]:
    if r["kind"] == "nearest":
        EDGE[r["site"]] = r
    elif r.get("contains"):
        CONT[r["site"]].append(r)
log(f"ในจาน {R_DISC} ม.: node มีแท็ก {len(nodes):,} · way ที่เก็บ {len(ways):,} · "
    f"relation {len(rels):,} · ผืน protection ที่ห่อจุด {sum(len(v) for v in CONT.values()):,} "
    f"· จุดที่ถูกห่อ {len(CONT)}/83")

# geometry จริง (ไม่ใช้ centroid) แล้ววัดระยะเป็นเมตรใน projection เดียวกับทั้งซีรีส์
recs = []
for w in ways:
    t = w["tags"]
    if not any(k in t for k in KEEP_GEOM):
        continue
    pts = wgeom.get(str(w["id"]))
    if not pts or len(pts) < 2:
        continue
    co = [(p[1], p[0]) for p in pts]
    g = None
    if len(pts) >= 4 and co[0] == co[-1]:
        try:
            g = Polygon(co)
            if not g.is_valid:
                g = make_valid(g)
        except Exception:
            g = None
    if g is None:
        g = LineString(co)
    if g.is_empty:
        continue
    recs.append(dict(id=w["id"], site=w["site"], cls="way", tags=t, geom=g,
                     closed=(g.geom_type == "Polygon")))
for n in nodes:
    recs.append(dict(id=n["id"], site=n["site"], cls="node", tags=n["tags"],
                     geom=Point(n["lon"], n["lat"]), closed=False))
Gm = gpd.GeoDataFrame(recs, geometry="geom", crs="EPSG:4326").to_crs(EQ)
Gm["tags"] = [r["tags"] for r in recs]
Gm["closed"] = [r["closed"] for r in recs]
log(f"วัตถุที่มี geometry พร้อมวัดระยะ: {len(Gm):,} ชิ้น")

tr = Transformer.from_crs("EPSG:4326", EQ, always_xy=True)
SP = np.array(tr.transform(P[:, 0], P[:, 1])).T
guard_crs([SP[:, 0].min(), SP[:, 1].min(), SP[:, 0].max(), SP[:, 1].max()],
          "darksky-projected-metres", expect="metres")
SITE_PT = {j: Point(SP[j][0], SP[j][1]) for j in range(83)}
Gm["d_site"] = [g.distance(SITE_PT[s]) for g, s in zip(Gm.geometry.values, Gm.site.values)]
Gm["inside"] = [(g.contains(SITE_PT[s]) if (c and g.geom_type == "Polygon") else False)
                 for g, s, c in zip(Gm.geometry.values, Gm.site.values, Gm.closed.values)]

## ตอน 2) จับคู่ชื่อ 4 การทดสอบ (เข้ม → อ่อน)

In [ ]:
# ================================================== 2. จับคู่ชื่อ 4 การทดสอบ (เข้ม → อ่อน)
NAME_FIELDS = ["name", "name:th", "name:en", "int_name", "official_name",
               "alt_name", "old_name", "loc_name", "short_name"]

def name_forms(t):
    out = []
    for f in NAME_FIELDS:
        v = t.get(f)
        if isinstance(v, str) and v.strip():
            for part in re.split(r"[;,/]", v):
                if part.strip():
                    out.append((f, part.strip()))
    return out

def grade(reg, cand):
    """1 = พิสูจน์ได้จากชื่อ · 2 = คล้ายจนควรตรวจด้วยตา · 0 = ไม่ตรง"""
    k_th, k_lat = nk(cand), la(cand)
    if not k_th and not k_lat:
        return 0, "", ""
    if k_th and k_th == reg.key_th:
        return 1, "ชื่อไทยตรงกันทุกตัวอักษร", "full"
    if k_lat and reg.key_lat and (k_lat == reg.key_lat or
                                  (min(len(k_lat), len(reg.key_lat)) >= 6 and
                                   (k_lat in reg.key_lat or reg.key_lat in k_lat))):
        return 1, "รูปละตินของ slug ทางการตรงกับชื่อใน OSM", "full"
    a = {w for w in latin_words(cand) if w not in GENERIC}
    b = reg.slug_tok
    shared = len(a & b)
    # ต้องมีคำหลักตรงกันอย่างน้อย 2 คำ (หรือคำเดียวถ้าสองข้างมีคำเดียวจริง ๆ)
    # กันคำเดียว (เช่น "phu") ทำให้สถานที่คนละแห่งดูคล้ายกัน
    if shared and (shared >= 2 or (shared == 1 and len(a) == 1 and len(b) == 1)) and \
       shared / max(1, min(len(a), len(b))) >= 0.5:
        return 2, "คำหลักละตินตรงกันเกินครึ่ง (สะกดต่างกัน)", "tok"
    r1 = difflib.SequenceMatcher(None, reg.key_th, k_th).ratio() if k_th else 0.0
    r2 = difflib.SequenceMatcher(None, reg.key_lat, k_lat).ratio() if (k_lat and reg.key_lat) else 0.0
    if max(r1, r2) >= 0.78:
        return 2, f"คล้ายตามลำดับตัวอักษร {max(r1, r2):.2f}", "sim"
    return 0, "", ""

REJ = []
EMPTY = dict(lvl=0, osm_name="", field="", why="", dist=np.nan, cls="", inside=False,
             osm_id=None, tourism="", amenity="", tags={})
best = {j: dict(EMPTY) for j in range(83)}
for j, reg in D.iterrows():
    sub = Gm[(Gm.site == j) & (Gm.d_site <= R_MATCH)]
    for i in sub.index:
        t = Gm.tags.at[i]
        for f, cand in name_forms(t):
            lvl, why, kind = grade(reg, cand)
            if lvl == 0:
                continue
            cap = {1: OK_FULL, 2: (OK_SIM if kind == "sim" else OK_TOK)}[lvl]
            d = float(Gm.d_site.at[i])
            if d > cap:
                REJ.append(dict(site=int(j), registry=reg["name"], osm_name=cand,
                                dist_m=round(d, 1), lvl=lvl, kind=kind, why=why,
                                cap_m=cap))
                continue                     # ชื่อคล้ายแต่ไกลเกินหลักฐานสนับสนุน = คนละสถานที่
            cur = best[j]
            if (lvl > cur["lvl"]) or (lvl == cur["lvl"] and d < cur["dist"]):
                best[j] = dict(lvl=lvl, osm_name=cand, field=f, why=why,
                               dist=d, cls=Gm.cls.at[i],
                               inside=bool(Gm.inside.at[i]), osm_id=int(Gm.id.at[i]),
                               tourism=t.get("tourism", "") or "",
                               amenity=t.get("amenity", "") or "", tags=t)

best_poi = {j: dict(v) for j, v in best.items()}   # สถานะก่อนรวมผืน protection (ใช้เทียบผล)

# ----------------------------------------------- ผืน protection (relation/way ใหญ่) ที่ห่อจุด
# ขอบอุทยานมักอยู่ไกลกว่าจาน 2 กม. จึงไม่โผล่มาเป็น node/way — ถ้า OSM วาดผืนห่อจุดทะเบียนไว้
# และชื่อผืนชนกับชื่อทะเบียน ถือว่า "รู้จัก" เช่นกัน (ระยะ = 0 เพราะจุดอยู่ *ใน* ผืน)
rel_hit = {}
for j in range(83):
    for r in CONT.get(j, []):
        lvl, why, kind = grade(D.loc[j], r["name"])
        if lvl == 0:
            continue
        cap = OK_FULL if lvl == 1 else 600          # จุดอยู่ในผืน = ระยะ 0 ผ่านทุกเพดาน
        if 0 > cap:
            continue
        cur = best[j]
        if lvl > cur["lvl"]:
            best[j] = dict(lvl=lvl, osm_name=r["name"], field="area", why=f"{why} (ในผืนที่ OSM วาด)",
                           dist=0.0, cls="polygon", inside=True, osm_id=int(r["id"]),
                           tourism=r["keys"].get("tourism", ""), amenity="",
                           tags={k: v for k, v in r["keys"].items() if k in NAME_FIELDS})
    if CONT.get(j):
        rel_hit[j] = max((grade(D.loc[j], r["name"])[0] for r in CONT[j]), default=0)
n_poly_match = sum(1 for j in best if best[j]["cls"] == "polygon")
n_cont = len(CONT)
log(f"จับคู่จาก node/way/polygon: {sum(1 for j in best if best[j]['lvl'] > 0)} จุด · "
    f"ในจำนวนนี้ {n_poly_match} จุดจับคู่ได้จาก *ขอบผืน* เท่านั้น")

M = pd.DataFrame([dict(idx=j, lvl=v["lvl"], osm_name=v["osm_name"], field=v["field"],
                       why=v["why"], dist=v["dist"], cls=v["cls"], inside=v["inside"],
                       osm_id=v["osm_id"], tourism=v["tourism"], amenity=v["amenity"])
                  for j, v in best.items()])
M = M.merge(D[["name", "slug", "province", "region_th", "type"]], left_on="idx", right_index=True)
M["in_protected"] = M.idx.map(lambda j: bool(CONT.get(j)))
M["protected_name"] = M.idx.map(lambda j: max(CONT.get(j, []),
                                              key=lambda r: r["area_km2"])["name"] if CONT.get(j) else "")
M["protected_km2"] = M.idx.map(lambda j: max([r["area_km2"] for r in CONT.get(j, [])],
                                             default=None))
M["edge_m"] = M.idx.map(lambda j: EDGE.get(j, {}).get("dist_edge_m"))

## ตอน 3) หกระดับ "OSM รู้จักที่นี่แค่ไหน"

In [ ]:
# ================================================== 3. หกระดับ "OSM รู้จักที่นี่แค่ไหน"
in_match = Gm[Gm.d_site <= R_MATCH]
stay_near = in_match[in_match.tags.map(lambda t: (t.get("tourism") in STAY or
                                                  t.get("amenity") in STAY))].groupby("site").size()
M["n_elem"] = M.idx.map(in_match.groupby("site").size()).fillna(0).astype(int)
M["n_stay"] = M.idx.map(stay_near).fillna(0).astype(int)
M["n_disc"] = M.idx.map(Gm.groupby("site").size()).fillna(0).astype(int)

def cls_of(r):
    if r.lvl == 1:
        return "1-ชื่อตรงกัน"
    if r.lvl == 2:
        return "2-ชื่อใกล้เคียง"
    if r.in_protected and r.lvl == 0:
        return "2n-อยู่ในผืน OSM แต่ชื่อไม่ตรง"
    if r.n_stay >= 1:
        return "3-วัตถุประเภทเดียวกัน คนละชื่อ"
    if r.n_elem >= 1:
        return "4-มีอย่างอื่น ไม่ใช่ที่นี่"
    return "5-ไม่มีวัตถุเลย"
M["klass"] = M.apply(cls_of, axis=1)
ORDER = ["1-ชื่อตรงกัน", "2-ชื่อใกล้เคียง", "2n-อยู่ในผืน OSM แต่ชื่อไม่ตรง",
         "3-วัตถุประเภทเดียวกัน คนละชื่อ", "4-มีอย่างอื่น ไม่ใช่ที่นี่", "5-ไม่มีวัตถุเลย"]
counts = {k: int((M.klass == k).sum()) for k in ORDER}
named = counts[ORDER[0]] + counts[ORDER[1]]
log(" · ".join(f"{k}={counts[k]}" for k in ORDER))
assert sum(counts.values()) == 83, "หกระดับต้องรวมกันครบ 83"

ok = M[M.lvl >= 1].copy()
# ระยะคลาดวัดได้เฉพาะคู่ที่เป็น "วัตถุจุด/เส้น" (POI) เพราะคู่จากผืน = 0 โดยนิยาม
ok_poi = ok[ok.cls != "polygon"].copy()
assert ok.dist.notna().all()
guard_variance(ok_poi.dist.values, "registry-vs-osm-poi-distance")

by_type = (M.groupby("type").agg(sites=("idx", "size"),
                                 named=("lvl", lambda s: int((s >= 1).sum())),
                                 in_prot=("in_protected", lambda s: int(s.sum())),
                                 elem_med=("n_elem", "median")).reset_index())
by_type["named_pct"] = (by_type.named / by_type.sites * 100).round(1)

## ตอน 4) ถนนที่ OSM รู้จัก (วัดถึงขอบจริง)

In [ ]:
# ================================================== 4. ถนนที่ OSM รู้จัก (วัดถึงขอบจริง)
HWY = {}
for w in ways:
    hv = w["tags"].get("highway")
    if not hv:
        continue
    pts = wgeom.get(str(w["id"]))
    if not pts or len(pts) < 2:
        continue
    co = [(p[1], p[0]) for p in pts]
    g = Polygon(co).exterior if (co[0] == co[-1] and len(co) >= 4) else LineString(co)
    HWY.setdefault(w["site"], []).append((g, hv, w["tags"].get("name", ""), w["id"]))
hrows = [dict(site=s, highway=h, name=nm, id=i, geometry=g)
         for s, lst in HWY.items() for g, h, nm, i in lst]
hgw = gpd.GeoDataFrame(hrows, geometry="geometry", crs="EPSG:4326").to_crs(EQ)
rr = []
for j in range(83):
    sub = hgw[hgw.site == j]
    if not len(sub):
        rr.append(dict(idx=j, road_n=0, road_min_m=np.nan, road_class="", road_name=""))
        continue
    d = sub.geometry.distance(SITE_PT[j])
    k = int(d.idxmin())
    rr.append(dict(idx=j, road_n=int(len(sub)), road_min_m=round(float(d.min()), 1),
                   road_class=str(sub.loc[k, "highway"]), road_name=str(sub.loc[k, "name"])))
M = M.merge(pd.DataFrame(rr), on="idx")
rd = M.road_min_m.dropna()
log(f"จุดที่มีถนนในจาน: {int((M.road_n > 0).sum())}/83 · มัธยฐาน {rd.median():,.0f} ม. · "
    f"ไกลสุดที่วัดได้ {rd.max():,.0f} ม.")

## ตอน 5) สองแหล่งรู้เรื่องเดียวกันไหม (ทีละช่อง)

In [ ]:
# ================================================== 5. สองแหล่งรู้เรื่องเดียวกันไหม (ทีละช่อง)
OSM_FIELDS = [("เบอร์โทร", ["phone", "contact:phone"], "ทะเบียนมีช่อง contact"),
              ("ที่อยู่", ["addr:full", "addr:housenumber", "addr:street", "addr:city"],
               "ทะเบียนมีช่อง address"),
              ("เวลาเปิดทำการ", ["opening_hours"], "ทะเบียนไม่มีช่องนี้"),
              ("เว็บไซต์/เพจ", ["website", "contact:website"], "ทะเบียนมีแต่ URL ของ NARIT"),
              ("ชื่อภาษาอังกฤษ", ["name:en", "int_name"], "ทะเบียนไม่มี แต่ slug จาก URL ก็เป็นละติน"),
              ("ระบุตัวด้วย Wikidata", ["wikidata"], "ทะเบียนไม่มีช่องนี้"),
              ("อินเทอร์เน็ต", ["internet_access"], "ทะเบียนไม่มีช่องนี้"),
              ("ห้องน้ำ", ["toilets"], "ทะเบียนไม่มีช่องนี้")]
FIELD_ROWS = []
for nm, keys, note in OSM_FIELDS:
    has = np.zeros(83, bool); reg_has = np.zeros(83, bool)
    for j in range(83):
        if best[j]["lvl"] >= 1:
            has[j] = any(str(best[j]["tags"].get(k, "")).strip() not in ("", "None") for k in keys)
    if nm == "เบอร์โทร":
        reg_has = D.contact.fillna("").astype(str).str.strip().ne("").values
    elif nm == "ที่อยู่":
        reg_has = D.address.fillna("").astype(str).str.strip().ne("").values
    elif nm in ("ชื่อภาษาอังกฤษ", "เว็บไซต์/เพจ"):
        reg_has[:] = True
    both = int((reg_has & has).sum()); r_only = int((reg_has & ~has).sum())
    o_only = int((~reg_has & has).sum()); neither = int((~reg_has & ~has).sum())
    FIELD_ROWS.append(dict(field=nm, registry=int(reg_has.sum()), osm=int(has.sum()),
                           both=both, registry_only=r_only, osm_only=o_only,
                           neither=neither, note=note))
FL = pd.DataFrame(FIELD_ROWS)
log("ช่องที่ OSM รู้ (เฉพาะจุดที่จับคู่ได้): " + ", ".join(f"{r.field} {r.osm}" for r in FL.itertuples()))
keyc = Counter()
for j in range(83):
    if best[j]["lvl"] >= 1:
        keyc.update(best[j]["tags"].keys())
TOP_KEYS = keyc.most_common(14)

## ตอน 6) สัมพันธ์กับกิจกรรม map (Day 16)

In [ ]:
# ================================================== 6. สัมพันธ์กับกิจกรรม map (Day 16)
d16 = pd.read_csv(os.path.join(WORK, "out", "day16", "day16_sites_osm.csv"))
J = M.merge(d16[["idx", "ways", "users", "urban_pop_30km", "holder"]], on="idx", how="left")
assert len(J) == 83 and J.ways.notna().all(), "ต้องต่อกับ Day 16 ครบ 83 จุด"
# คะแนน = หลักฐานเรื่อง *ชื่อ* เท่านั้น (0 ไม่มี / 1 พิสูจน์ได้ / 2 ใกล้เคียง)
# ไม่นำ "อยู่ในผืนที่ OSM วาด" มารวมเป็นอันดับ เพราะเป็นของอีกมิติ (วัดแยกเป็น in_protected)
J["score"] = J.klass.map({ORDER[0]: 5, ORDER[1]: 4, ORDER[2]: 3, ORDER[3]: 2,
                          ORDER[4]: 1, ORDER[5]: 0})
assert J.score.notna().all(), "ทุกระดับ klass ต้อง map เป็นคะแนน"
J["lvl_poi"] = J.idx.map(lambda j: best_poi[j]["lvl"])
rho_w, p_w = spearmanr(J.score.values, J.ways.values)
rho_lvl, p_lvl = spearmanr(J.lvl.values, J.ways.values)
rho_pre, p_pre = spearmanr(J.lvl_poi.values, J.ways.values)
rho_poi, p_poi = spearmanr(J.score.values, J.n_elem.values)
rho_c, p_c = spearmanr(J.score.values, np.log1p(J.urban_pop_30km.values))
rho_r, p_r = spearmanr(J.score.values, J.road_min_m.fillna(R_DISC).values)
byl = (J.groupby("lvl").agg(sites=("idx", "size"), ways_med=("ways", "median"),
                            users_med=("users", "median"), pop_med=("urban_pop_30km", "median"),
                            elem_med=("n_elem", "median")).reset_index())
log(f"Spearman(อันดับ 0-5 จาก klass, ways Day 16) = {rho_w:.3f} (p = {p_w:.3g}) · "
    f"(lvl รวมผืน = {rho_lvl:.3f}, p {p_lvl:.3g}) · ก่อนรวมผืน = {rho_pre:.3f} (p {p_pre:.3g}) · "
    f"ประชากรเมือง = {rho_c:.3f} (p {p_c:.3g})")

## ตอน 7) คิอแก้ — เตรียมไว้ ยังไม่ได้ส่ง

In [ ]:
# ================================================== 7. คิอแก้ — เตรียมไว้ ยังไม่ได้ส่ง
q = M[M.lvl < 1].copy()
q["phone_from_registry"] = [str(D.loc[i, "contact"]).replace("โทร:", "").strip() for i in q.idx]
q["suggestion"] = np.where(
    q.in_protected,
    "OSM วาดขอบผืนห่อจุดนี้ไว้แล้ว แต่ไม่มีชื่อที่ตรงกับทะเบียน — เพิ่ม name ลงบนผืนนั้น",
    np.where(q.n_stay > 0,
             "มีวัตถุ tourism/amenity ใกล้ ๆ คนละชื่อ ตรวจว่าคือที่นี่ไหม ถ้าใช่เพิ่ม name + name:en",
             np.where(q.n_elem > 0,
                      "มีวัตถุอื่นในระยะแต่ไม่ใช่ที่นี่ ถ้าจะเพิ่มจุดใหม่ต้องใช้ข้อมูลที่ยืนยันเอง",
                      "ไม่พบวัตถุใดในระยะ 1,200 ม. ถ้าจะเพิ่ม ต้องมีคนยืนยันภาคพื้น/ภาพถ่ายก่อน")))
def mask_phone(v):
    """เบอร์โทรจริงไม่ลงใน repo — OSM เองก็ Require การยืนยันจากภาคพื้นที่ก่อนใส่ phone
    ไฟล์เต็ม (ไม่ mask) ยังอยู่ใน out/day21/ สำหรับคนที่ต้องมีตัวเลขไปโทรสอบถาม"""
    digits = re.findall(r"[0-9]+", str(v))
    return (digits[0][:3] + "-xxx-xxxx") if digits and digits[0].startswith("0") else ""

SHIP_PHONES = False
EDIT = q[["idx", "name", "slug", "province", "region_th", "type", "klass", "n_elem",
          "n_stay", "road_min_m", "road_class", "phone_from_registry", "suggestion"]].copy()
if not SHIP_PHONES:
    EDIT["phone_from_registry"] = [mask_phone(v) for v in EDIT["phone_from_registry"]]
# ไฟล์ขึ้นต้นด้วย _ = finalize.py ข้าม (เบอร์จริงอยู่เฉพาะในเครื่อง ไม่ลง repo)
q.to_csv(os.path.join(OUT, "_day21_edit_queue_with_phones_LOCAL.csv"), index=False)
log(f"คิอแก้ {len(EDIT)} จุด · สคริปต์นี้ไม่ล็อกอิน ไม่ส่ง changeset ใด ๆ")

## ตอน 8) ภาพ

In [ ]:
# ================================================== 8. ภาพ
COL = {ORDER[0]: TEAL, ORDER[1]: COOL, ORDER[2]: PURPLE, ORDER[3]: ACCENT,
       ORDER[4]: MUTED, ORDER[5]: WARM}
ne = gpd.read_file(os.path.join(FONTS, "ne_10m_admin_0_countries.shp"))
TH = ne[ne.NAME == "Thailand"].geometry.iloc[0]
thp = gpd.GeoSeries([make_valid(TH)], crs="EPSG:4326").to_crs(EQ).iloc[0]

def draw_hero():
    fig = plt.figure(figsize=(14.8, 8.8), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.02, 0.025, 0.60, 0.865]); ax.set_facecolor(BG)
    for g in (thp.geoms if thp.geom_type == "MultiPolygon" else [thp]):
        x, y = g.exterior.xy
        ax.fill(x, y, color="#111c2b", zorder=0)
        ax.plot(x, y, color="#2b3b57", lw=0.9, zorder=1)
    ring_draw = 0
    for poly_q in poly_draw:
        rr = np.array(poly_q["ring"])
        gx, gy = tr.transform(rr[:, 0], rr[:, 1])
        ax.plot(gx, gy, color="#4a6b4f", lw=0.8, alpha=0.9, zorder=2)
        ring_draw += 1
    seg = 0
    for lst in HWY.values():
        for g, h, nm, i in lst:
            x, y = g.xy
            ax.plot(np.asarray(x), np.asarray(y), color="#3b5c86", lw=0.3, alpha=0.8, zorder=2)
            seg += 1
    for k in reversed(ORDER):
        s = M[M.klass == k]
        ax.scatter([SITE_PT[j].x for j in s.idx], [SITE_PT[j].y for j in s.idx], s=44,
                   color=COL[k], edgecolor=BG, lw=0.6, zorder=5,
                   label=f"{k} · {counts[k]} จุด")
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])      # หน่วยเมตรบนAxes นี้ไร้ความหมาย ทำให้legend ทับ
    for sp in ax.spines.values(): sp.set_color("#33415e")
    ax.set_title(T(f"83 แหล่งดูฟ้ามืดที่ราชการรับรอง กับสิ่งที่ OpenStreetMap เก็บไว้\n"
                   f"เส้นเขียว = ขอบพื้นที่คุ้มครองที่ OSM วาด ({ring_draw:,} วง) "
                   f"เส้นน้ำเงิน = ถนนในจาน {R_DISC:,} ม. ({seg:,} เส้น) "
                   f"จุดสี = ระดับที่ OSM รู้จักชื่อนั้น"), color=INK, fontsize=10.5, pad=10)
    leg = ax.legend(loc="lower left", facecolor="#111c2b", edgecolor="#33415e",
                    labelcolor=INK, fontsize=9.2, framealpha=0.95)
    leg.set_title(T("ระดับการรู้จัก"), prop={"size": 10, "family": "Trirong"})
    ax.text(0.01, -0.055, "ข้อมูล © OpenStreetMap contributors (ODbL) ผ่าน Geofabrik "
            "Thailand extract · ขอบประเทศ Natural Earth 10m", transform=ax.transAxes,
            color=MUTED, fontsize=8.2)
    txt = fig.add_axes([0.645, 0.025, 0.345, 0.865]); txt.axis("off"); txt.set_facecolor(BG)
    lines = [(T("แหล่งในทะเบียน"), "83"),
             (T("OSM มีชื่อตรงกัน (ระดับ 1)"), f"{counts[ORDER[0]]}"),
             (T("OSM มีชื่อใกล้เคียง (ระดับ 2)"), f"{counts[ORDER[1]]}"),
             (T("รวมที่ OSM รู้ชื่อ"), f"{named} ({named / 83 * 100:.1f}%)"),
             (T("อยู่ในผืน protection แต่ชื่อไม่ตรง"), f"{counts[ORDER[2]]}"),
             (T("วัตถุประเภทเดียวกัน คนละชื่อ"), f"{counts[ORDER[3]]}"),
             (T("มีอย่างอื่น ไม่ใช่ที่นี่"), f"{counts[ORDER[4]]}"),
             (T("ไม่มีวัตถุเลยใน 1,200 ม."), f"{counts[ORDER[5]]}"),
             ("", ""),
             (T("จุดที่อยู่ในผืน protection"), f"{n_cont}"),
             (T("  จับชื่อได้จากชื่อผืนเท่านั้น"), f"{AGG['polygon_matches']}"),
             (T("  จุดทะเบียนใน polygon ที่เป็น POI"), f"{AGG['inside_poi_sites']}"),
             ("", ""),
             (T("วัตถุใน 1,200 ม. ต่อจุด"), f"มัธยฐาน {M.n_elem.median():.0f} · สูงสุด {int(M.n_elem.max())}"),
             (T(f"ระยะ POI ที่จับได้ ({len(ok_poi)} คู่)"),
              f"{ok_poi.dist.min():,.0f} / {ok_poi.dist.median():,.0f} / {ok_poi.dist.max():,.0f} ม."),
             (T("จับชื่อได้แต่ไกลกว่า 300 ม."), f"{int((ok_poi.dist > R_NEAR).sum())}"),
             ("", ""),
             (T("มีถนนในจาน 2 กม. · ไม่มี"), f"{int((M.road_n > 0).sum())} · {int((M.road_n == 0).sum())}"),
             (T("ถนนใกล้สุด (วัดถึงขอบ)"),
              f"มัธยฐาน {rd.median():,.0f} · ไกลสุด {rd.max():,.0f} ม."),
             (T("ขอบผืน protection ไกลจากจุด"),
              f"มัธยฐาน {AGG['edge_median_m']:,.0f} · ไกลสุด {AGG['edge_max_m']:,.0f} ม."),
             ("", ""),
             (T("Spearman อันดับรู้จักกับ ways"), f"{rho_w:.3f} (p {p_w:.3g})"),
             (T("  ก่อนแก้บั๊กผืน (lvl เท่านั้น)"), f"{rho_pre:.3f} (p {p_pre:.3g})"),
             (T("Spearman อันดับรู้จักกับประชากรเมือง"), f"{rho_c:.3f} (p {p_c:.3g})"),
             ("", ""),
             (T("คิอแก้ที่ยังไม่ได้ส่ง"), f"{len(EDIT)} จุด")]
    yy = 0.99
    for k, v in lines:
        if not k:
            yy -= 0.021; continue
        txt.text(0, yy, k, color=MUTED, fontsize=9.2, va="top")
        txt.text(1, yy, v, color=ACCENT, fontsize=9.2, va="top", ha="right")
        yy -= 0.0322
    ex = ok[ok.cls != "polygon"].sort_values("dist").head(4)
    txt.text(0, yy, T("ตัวอย่างชื่อที่ OSM ใช้"), color=INK, fontsize=10.2)
    yy -= 0.022
    for _, r in ex.iterrows():
        txt.text(0, yy, f"{str(r['name'])[:20]} → {str(r.osm_name)[:26]} "
                        f"({r.dist:,.0f} ม.)", color=MUTED, fontsize=8.4, va="top")
        yy -= 0.020
    fig.suptitle(T("ทะเบียนราชการ 83 แห่ง กับแหล่งข้อมูลมหัศจรรย์"), color=INK,
                 fontsize=15.5, y=0.985)
    p = os.path.join(OUT, "day21_two_sources.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

def draw_coverage():
    fig = plt.figure(figsize=(13.4, 6.9), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.055, 0.13, 0.40, 0.74]); ax.set_facecolor("#111c2b")
    x = np.arange(len(ORDER)); v = [counts[k] for k in ORDER]
    ax.bar(x, v, color=[COL[k] for k in ORDER], width=0.68)
    for i, val in enumerate(v):
        ax.text(i, val + max(v) * 0.02, str(val), color=INK, ha="center", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([k.split("-", 1)[1] for k in ORDER], color=INK, fontsize=8.2,
                       rotation=22, ha="right")
    ax.set_ylabel(T("จำนวนจุดทะเบียน"), color=INK); ax.set_ylim(0, max(v) * 1.2)
    ax.set_title(T("83 จุด แยกตามสิ่งที่ OSM เก็บไว้\nสามแท่งซ้าย = OSM รู้จักชื่อของที่นี่"),
                 color=INK, fontsize=12)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5, axis="y")
    ax.tick_params(colors=INK, labelsize=9)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color("#33415e")

    ax2 = fig.add_axes([0.55, 0.17, 0.42, 0.68]); ax2.set_facecolor("#111c2b")
    y = np.arange(len(FL))
    ax2.barh(y - 0.18, FL.registry, height=0.34, color=COOL, label=T("ทะเบียนราชการมีข้อมูล"))
    ax2.barh(y + 0.18, FL.osm, height=0.34, color=ACCENT,
             label=T("OSM มีข้อมูล (เฉพาะจุดที่จับคู่ได้)"))
    ax2.set_yticks(y); ax2.set_yticklabels(FL.field, color=INK, fontsize=9.2)
    for i, r in FL.iterrows():
        ax2.text(r.registry + 1, i - 0.18, str(r.registry), color=COOL, fontsize=8.4, va="center")
        ax2.text(r.osm + 1, i + 0.18, str(r.osm), color=ACCENT, fontsize=8.4, va="center")
    ax2.set_xlabel(T("จำนวนจุด (จาก 83) ที่ \"รู้\" ช่องนี้"), color=INK)
    ax2.set_xlim(0, 96)
    ax2.set_title(T("คำถามเดียวกัน สองฐานข้อมูลตอบไม่เท่ากัน\n"
                    "แถวที่แท่งส้มยาวกว่า = ช่องที่ทะเบียนไม่มีให้กรอก"), color=INK, fontsize=12)
    fig.legend(handles=ax2.get_legend_handles_labels()[0],
               labels=ax2.get_legend_handles_labels()[1], loc="lower right",
               bbox_to_anchor=(0.975, 0.012), facecolor="#111c2b", edgecolor="#33415e",
               labelcolor=INK, fontsize=8.6, framealpha=0.96)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    ax2.tick_params(colors=INK, labelsize=9)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    fig.suptitle(T("หนึ่งสถานที่ สองฐานข้อมูล ใครรู้อะไร"), color=INK, fontsize=15, y=0.975)
    p = os.path.join(OUT, "day21_fields.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

def draw_roads():
    fig = plt.figure(figsize=(13.8, 6.9), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.055, 0.12, 0.42, 0.75]); ax.set_facecolor("#111c2b")
    bins = np.arange(0, 2100, 150)
    ax.hist(rd.values, bins=bins, color=TEAL, edgecolor=BG, lw=0.4)
    ax.axvline(float(rd.median()), color=WARM, ls="--", lw=1.3)
    ax.text(float(rd.median()) + 60, ax.get_ylim()[1] * 0.97,
            T(f"มัธยฐาน {rd.median():,.0f} ม."), color=WARM, fontsize=10)
    ax.set_xlabel(T("ระยะจากจุดทะเบียนถึง *ถนนที่ OSM วาดไว้* ใกล้ที่สุด (ม.)"), color=INK)
    ax.set_ylabel(T("จำนวนจุด"), color=INK)
    ax.set_title(T(f"OSM มีถนนในจาน 2 กม. ของ {int((M.road_n > 0).sum())}/83 จุด\n"
                   f"ไม่มีถนนที่ OSM วาดเลยในระยะนั้น: {int((M.road_n == 0).sum())} จุด"),
                 color=INK, fontsize=12)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5, axis="y")
    ax.tick_params(colors=INK, labelsize=9)
    for s in ("top", "right"): ax.spines[s].set_visible(False)

    ax2 = fig.add_axes([0.545, 0.12, 0.42, 0.75]); ax2.set_facecolor("#111c2b")
    cc = M[M.road_n > 0].road_class.value_counts().head(9)
    ax2.barh(np.arange(len(cc)), cc.values, color=PURPLE, height=0.66)
    ax2.set_yticks(np.arange(len(cc)))
    ax2.set_yticklabels(list(cc.index), color=INK, fontsize=9.2)
    for i, (k, val) in enumerate(cc.items()):
        ax2.text(val + 0.35, i, str(int(val)), color=INK, fontsize=9, va="center")
    ax2.set_xlabel(T("จำนวนจุดที่ถนนใกล้ที่สุดเป็นประเภทนั้น"), color=INK)
    ax2.set_title(T("ถนนที่ OSM รู้จักใกล้จุดทะเบียนเป็นประเภทอะไร\n"
                    "(ค่า tag `highway` ของ way ที่ใกล้ที่สุด)"), color=INK, fontsize=12)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    ax2.set_xlim(0, max(cc.values) * 1.18)
    ax2.tick_params(colors=INK, labelsize=9)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    fig.suptitle(T("การเดินทางไปดูดาวในสายตา OSM"), color=INK, fontsize=15, y=0.975)
    p = os.path.join(OUT, "day21_roads.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 9) artifact + ผู้ตัดสิน

In [ ]:
# ================================================== 9. artifact + ผู้ตัดสิน
def sci(x):
    if x == 0 or not np.isfinite(x):
        return dict(mantissa=0.0, exponent=0, exponent_abs=0)
    s = f"{x:.2g}"
    if "e" not in s:
        return dict(mantissa=float(s), exponent=0, exponent_abs=0)
    m, e = s.split("e")
    return dict(mantissa=float(m), exponent=int(e), exponent_abs=abs(int(e)))

STATS_THEME = "The awesome data source for mapping. You can also use this day to edit OSM."
M.round(3).to_csv(os.path.join(OUT, "day21_sites_osm.csv"), index=False)
FL.to_csv(os.path.join(OUT, "day21_fields.csv"), index=False)
EDIT.to_csv(os.path.join(OUT, "day21_edit_queue.csv"), index=False)
gdf_out = Gm[["id", "site", "cls", "d_site", "inside", "geom"]].rename(columns={"geom": "geometry"}).assign(
    tags=[json.dumps(t, ensure_ascii=False) for t in Gm.tags],
    highway=[t.get("highway", "") for t in Gm.tags],
    tourism=[t.get("tourism", "") for t in Gm.tags],
    name=[t.get("name", "") or t.get("name:en", "") for t in Gm.tags])
gdf_out = gpd.GeoDataFrame(gdf_out, geometry="geometry", crs=EQ)
# ใส่ repo เฉพาะวัตถุที่มีชื่อหรือเป็น tourism (ถนนไร้ชื่อ 9,966 เส้นทำให้ไฟล์ 8.7 MB
# โดยไม่มีใครใช้ — ใครอยากได้ทั้งไฟล์อ่านจาก extract ได้ตามวิธีในโพสต์)
keep = gdf_out[(gdf_out["name"].astype(str).str.strip() != "") |
               (gdf_out["tourism"].astype(str).str.strip() != "")]
keep.to_crs("EPSG:4326").to_file(os.path.join(OUT, "day21_osm_named.geojson"), driver="GeoJSON")
log(f"geojson ที่ ship เข้า repo: {len(keep):,} จาก {len(gdf_out):,} วัตถุ (เฉพาะที่มีชื่อ/tourism)")

AGG = dict(sites=83, disc_m=R_DISC, match_m=R_MATCH, near_m=R_NEAR,
           elements_geometry=int(len(Gm)), roads_drawn=int(sum(len(v) for v in HWY.values())),
           **counts, named_total=int(named), named_pct=round(named / 83 * 100, 2),
           elem_median=float(M.n_elem.median()), elem_max=int(M.n_elem.max()),
           elem_zero=int((M.n_elem == 0).sum()),
           dist_n=int(len(ok_poi)),
           dist_min_m=round(float(ok_poi.dist.min()), 1),
           dist_median_m=round(float(ok_poi.dist.median()), 1),
           dist_mean_m=round(float(ok_poi.dist.mean()), 1),
           dist_max_m=round(float(ok_poi.dist.max()), 1),
           dist_over_near=int((ok_poi.dist > R_NEAR).sum()),
           inside_poi_sites=int((M.inside & (M.cls != "polygon")).sum()),
           road_sites=int((M.road_n > 0).sum()), road_none=int((M.road_n == 0).sum()),
           road_min_m=round(float(rd.min()), 1), road_median_m=round(float(rd.median()), 1),
           road_mean_m=round(float(rd.mean()), 1), road_max_m=round(float(rd.max()), 1),
           road_over_1000=int((rd > 1000).sum()),
           spearman_score_ways=round(float(rho_w), 3), p_score_ways=float(p_w),
           spearman_lvl_ways=round(float(rho_lvl), 3), p_lvl_ways=float(p_lvl),
           spearman_pre_poly_ways=round(float(rho_pre), 3), p_pre_poly_ways=float(p_pre),
           moved_by_polygon=int(sum(1 for j in best if best[j]["lvl"] > 0 and
                                    best_poi[j]["lvl"] == 0)),
           spearman_score_elems=round(float(rho_poi), 3), p_score_elems=float(p_poi),
           spearman_score_pop=round(float(rho_c), 3), p_score_pop=float(p_c),
           spearman_score_road=round(float(rho_r), 3), p_score_road=float(p_r),
           poly_matches=int(n_poly_match), sites_in_protected=int(n_cont),
           protected_name_match=int(sum(1 for j in CONT if best[j]["lvl"] >= 1
                                        and best[j]["cls"] == "polygon")),
           edge_min_m=round(float(np.nanmin([v["dist_edge_m"] for v in EDGE.values()])), 1),
           edge_median_m=round(float(np.nanmedian([v["dist_edge_m"] for v in EDGE.values()])), 1),
           edge_max_m=round(float(np.nanmax([v["dist_edge_m"] for v in EDGE.values()])), 1),
           poly_draw_rings=int(len(poly_draw)), edit_queue_rows=int(len(EDIT)),
           poly_records=int(len(poly["records"])),
           poly_contain_records=int(sum(1 for r in poly["records"] if r.get("contains"))),
           polygons_way=int(poly["polygons_way"]), polygons_relation=int(poly["polygons_relation"]),
           protected_relations_all=int(poly["protected_relations_all"]),
           disc_nodes=int(meta["counts"]["disc_nodes"]),
           named_geo_rows=int(len(keep)),
           node_matches=int((ok.cls == "node").sum()), way_matches=int((ok.cls == "way").sum()),
           polygon_matches=int((ok.cls == "polygon").sum()),
           counts_of_file=meta["counts"], scan_md5=meta["md5"], scan_read_utc=meta["read_utc"])
STATS = dict(
    theme=STATS_THEME, theme_name="OpenStreetMap",
    theme_source="30daymapchallenge.com หัวข้อ Day 21 'OpenStreetMap' (อ่านสด 19:12 น. วันที่ 21 พ.ย. 2569)",
    theme_fetch_time=dict(hour=19, minute=12, day=21, month=11, year=2569),
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    question="OSM เก็บชื่อและข้อมูลของ 83 แหล่งดูฟ้ามืดที่ราชการรับรองไว้แค่ไหน "
             "สองฐานข้อมูลนี้รู้เรื่องเดียวกันไม่เท่ากันตรงไหนบ้าง",
    provenance=dict(source=meta["source"], url=meta["url"], md5=meta["md5"], bytes=meta["bytes"],
                    pbf_mb=round(meta["bytes"] / 1e6, 1),
                    overpass_http_errors=[502, 504],
                    overpass_first_attempt=dict(hour=19, minute=15),
                    overpass_give_up=dict(hour=19, minute=30),
                    extract_update_hour_utc=1, extract_update_minute_utc=0,
                    day16_collect_day=17, day16_collect_month=11,
                    extract_updated=meta["last_modified_header"], read_utc=meta["read_utc"],
                    read_by="tools/collect_day21_pbf.py (pyosmium)",
                    registry="data/darksky_thailand.csv (ชุดเดียวกับ Day 3–20)",
                    attribution="© OpenStreetMap contributors (ODbL)",
                    api_attempt="เริ่มที่ยิง Overpass API (tools/collect_day21_osm.py) แต่ทุก mirror "
                                "ตอบ 502/504 ติดกัน 20+ นาที จึงย้ายมาอ่านไฟล์ extract ในเครื่อง"),
    thresholds=dict(similarity_min=0.78, token_overlap_min=0.5, cap_full_m=OK_FULL,
                    cap_similarity_m=OK_SIM, cap_token_m=OK_TOK, token_len_min=3,
                    token_generic_words=int(len(GENERIC)), thai_exact="ต้องเท่ากันหลัง NFKC + ตัดช่อง"),
    api_outage=dict(http_errors=[502, 504], endpoints_tried=4,
                    first_attempt=dict(hour=19, minute=15), abandoned=dict(hour=19, minute=30),
                    note="คืน 21 พ.ย. ทั้งสี่ mirror ตอบ 502/504 ต่อเนื่อง ย้ายไปอ่าน extract"),
    method=dict(disc=f"สแกน node/way/relation ที่แตะจานรัศมี {R_DISC} ม. รอบจุดทะเบียน 83 จุด",
                distance="ระยะวัดจากจุดทะเบียน (projection เมตร) ถึงขอบวัตถุที่ใกล้ที่สุด "
                         "ไม่ใช่จุดศูนย์กลางของ way อย่าง Day 16",
                matching="4 การทดสอบเรียงความเข้ม: (1) ชื่อไทยตรงกันทุกตัวอักษรหลัง NFKC + ตัดช่อง"
                         " (2) รูปละตินของ slug จาก URL ทางการตรงกับชื่อใน OSM "
                         " (3) คำหลักละตินตรงกันเกินครึ่งหลังตัดคำ GENERIC "
                         " (4) ความคล้ายลำดับตัวอักษร ≥ 0.78",
                relation_note="relation (ขอบอุทยาน/เขตอนุรักษ์) ที่ *ใหญ่กว่าจานสแกน* ไม่โผล่มา "
                              "ใน node/way ของ collect_day21_pbf.py จึงสแกนซ้ำด้วย "
                              "collect_day21_polygons.py แล้วถามว่าจุดทะเบียนอยู่ *ใน* ผืนไหน "
                              "(STRtree predicate=contains) — ชื่อของผืนถูกตรวจด้วย grade() "
                              "ฟังก์ชันเดียวกับที่ใช้ตรวจชื่อ POI",
                protected_areas="สแกน boundary=protected_area / leisure=nature_reserve|park / "
                                "tourism=national_park / landuse=forest ทั้งแบบ way และ relation "
                                "(784 relation ทั้งประเทศ + 13,769 way ที่เกี่ยวเนื่อง) "
                                "แล้วทดสอบ contains ต่อ 83 จุด",
                editing="อ่านไฟล์อย่างเดียว ไม่ล็อกอิน ไม่ส่ง changeset — คิอแก้คือ "
                        "day21_edit_queue.csv ให้คนตรวจก่อนเสมอ"),
    bug_found_and_fixed=dict(
        symptom="รันแรกจัด อุทยานแห่งชาติเขาใหญ่ (กับอีก 1 จุด) เป็น class 5 'ไม่มีวัตถุเลยใน 1,200 ม.'",
        cause="จานสแกน 2 กม. เก็บเฉพาะ node/way ที่แตะจาน ขอบอุทยานขนาดใหญ่จึงหลุดทั้งผืน "
              "การสรุปว่า 'OSM ไม่รู้จัก' จากพื้นที่ว่างในจานจึงเท็จ",
        fix="เพิ่ม collect_day21_polygons.py สแกน polygon ทั่วประเทศที่เกี่ยวกับการปกป้อง "
            "แล้วทดสอบ contains ด้วย STRtree",
        lesson="รัศมี search เป็นนิยามของคำถาม ไม่ใช่หลักฐานของการไม่มีอยู่"),
    classes=counts, by_type=by_type.to_dict("records"), fields=FL.to_dict("records"),
    score_vs_activity=byl.to_dict("records"), top_tag_keys=dict(TOP_KEYS),
    named_examples=[{k: v for k, v in r.items() if k in
                     ("name", "osm_name", "field", "why", "dist", "cls", "inside",
                      "province", "tourism", "amenity")}
                    for r in ok[ok.lvl == 1].sort_values("dist").to_dict("records")],
    near_matches=[{k: v for k, v in r.items() if k in
                   ("name", "osm_name", "why", "dist", "cls", "province")}
                  for r in ok[ok.lvl == 2].sort_values("dist").to_dict("records")],
    false_positives_rejected=sorted(REJ, key=lambda r: -r["dist_m"])[:12],
    rejection_counts=dict(total_rejected=int(len(REJ)),
                          sites_with_rejection=int(len({r["site"] for r in REJ})),
                          max_rejected_dist_m=round(max([r["dist_m"] for r in REJ]), 1) if REJ else 0.0),
    far_matches=[{k: v for k, v in r.items() if k in
                  ("name", "osm_name", "why", "dist", "cls", "province")}
                 for r in ok[ok.dist > R_NEAR].sort_values("dist", ascending=False)
                 .head(10).to_dict("records")],
    unmatched=[{k: v for k, v in r.items() if k in
                ("name", "province", "type", "klass", "n_elem", "n_stay", "road_min_m",
                 "road_class")}
               for r in M[M.lvl < 1].sort_values(["klass", "n_elem"], ascending=[True, False])
               .to_dict("records")][:24],
    road_class_top=M[M.road_n > 0].road_class.value_counts().head(9).to_dict(),
    aggregates=AGG,
    filter_probe=dict(relations_broad=len(_broad), relations_strict_protected_area=strict,
                      relations_missed=int(len(_broad) - strict), by_value=missed,
                      khao_yai=dict(id=int(KY["id"]), name=KY["name"], keys=KY["keys"],
                                    area_km2=KY["area_km2"]) if KY else None,
                      note="ตัวเลขทั้งบล็อกนับจาก prot_ways.json.gz (แคชของ pass A) ไม่ใช่ประมาณการ"),
    before_after_fix=dict(without_polygons=dict(rho=round(float(rho_pre), 3), p=round(float(p_pre), 4),
                            moved_sites=int(AGG["moved_by_polygon"])),
                          with_polygons=dict(rho=round(float(rho_lvl), 3), p=round(float(p_lvl), 4)),
                          note="ก่อนสแกนผืน protection มีจุด 10 แห่งที่ถูกหาว่า 'OSM ไม่รู้จัก' "
                               "ทั้งที่ OSM วาดผืนห่อไว้ และมีชื่อตรงกับทะเบียน — "
                               "เมื่อแก้แล้ว ความสัมพันธ์กับปริมาณ map ของ Day 16 ก็เปลี่ยนตาม"),
    p_value_notation=dict(score_vs_ways=sci(p_w), score_vs_pop=sci(p_c), score_vs_road=sci(p_r),
                          note="p-value เขียนเป็น mantissa × 10^exponent เพื่อให้ guard_claims "
                                "ตามตัวเลขได้"),
    cross_check_series=dict(
        day16_source="Overpass API · ทุก way ใน 1,000 ม. วัดจากจุดศูนย์กลางของ way",
        day16_ways_median=float(np.median(d16.ways)),
        day21_ways_in_disc_median=float(np.median([sum(1 for w in ways if w["site"] == j)
                                                   for j in range(83)])),
        day20_cells_reused="กริด H3 res 5 เดิมไม่ถูกใช้ในวันนี้ วันนี้วัดระยะจากขอบวัตถุจริง",
        equal_expected=False,
        note="สองวันนับคนละนิยาม (รัศมี 1 vs 2 กม. · จุดศูนย์กลาง vs แตะจาน · way ทุกชนิด vs "
             "way ที่มีแท็กน่าสนใจ) จึงใช้เทียบแนวโน้ม ไม่ได้เทียบค่า"),
    figures=["day21_two_sources.png", "day21_fields.png", "day21_roads.png"],
    metric_definitions={
        "lvl": "1 = ชื่อพิสูจน์ได้ (ไทยตรงกัน หรือรูปละตินของ slug ตรงกับชื่อ OSM) · "
               "2 = ใกล้เคียง (คำหลักเกินครึ่ง หรือความคล้าย ≥ 0.78) · 0 = ไม่จับคู่กับ node/way",
        "klass": " || ".join(f"{k}: {counts[k]}" for k in ORDER),
        "in_protected": "จริงเมื่อจุดทะเบียนอยู่ภายใน polygon ที่ OSM วาดเป็นพื้นที่คุ้มครอง "
                        "(boundary=protected_area / leisure=nature_reserve / park / "
                        "tourism=national_park / landuse=forest) — วัดจาก geometry ของผืน "
                        "ไม่ขึ้นกับรัศมีจาน จึงต่างจาก class 4/5 โดยสิ้นเชิง",
        "edge_m": "ระยะจากจุดทะเบียนถึงขอบ *พื้นที่คุ้มครองที่ OSM วาด* ใกล้ที่สุด (เมตร)",
        "polygon match": "จับชื่อได้กับชื่อของผืนที่ห่อจุดทะเบียน (OSM รู้จักในฐานะพื้นที่ "
                         "ไม่ใช่ POI) — รายงาน dist = 0 เพราะจุดอยู่ภายในผืน",
        "dist": f"ระยะจากจุดทะเบียนถึง *ขอบ* วัตถุที่จับคู่ได้ (เมตร) เฉพาะวัตถุภายใน {R_MATCH} ม.",
        "n_elem": f"จำนวนวัตถุ OSM ที่มีแท็กน่าสนใจภายใน {R_MATCH} ม. จากจุดทะเบียน",
        "n_stay": f"จำนวนวัตถุ tourism/amenity ในหมวดที่พัก/แหล่งเที่ยวภายใน {R_NEAR} ม.",
        "road_min_m": f"ระยะจากจุดทะเบียนถึง way ที่เป็น highway ใกล้เคียงที่สุดภายในจาน {R_DISC} ม.",
        "inside": "จริงเมื่อจุดทะเบียนอยู่ภายใน polygon ของวัตถุที่ OSM วาด (ที่นี่ถูกวาดเป็นขอบเขต)"},
    limitations=["การจับคู่ชื่อไทย-ละตินเป็น heuristics มีพลาดได้เมื่อสะต่างกันคนละเรื่อง "
                 "จึงเปิดตัวอย่างที่จัดไว้ใน named_examples/near_matches ให้ตรวจตาม",
                 "relation เป็นขอบเขต ไม่ใช่จุด จึงไม่เอามาวัดระยะคลาด",
                 "จาน 2 กม. ไม่ได้แปลว่าไกลกว่านั้นไม่มีถนน — 'ไม่มี' ในที่นี้คือ OSM ไม่ได้วาดไว้ภายใน 2 กม.",
                 "OSM ไม่ใช่ทะเบียนสถานที่ การไม่พบไม่ได้แปลว่าผิด "
                 "มันคือภาพว่าชุมชน mapper ยังไปบันทึกไม่ถึง"],
    note="ภาพนี้ไม่มี tile ภายนอก: เส้นถนน/ขอบวัตถุทั้งหมดวาดจาก geometry ในไฟล์ extract "
         "ของ OSM เอง ขอบประเทศใช้ Natural Earth 10m")
import math as _mm
def _clean(o):
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (float, np.floating)):
        f = float(o)
        return None if (_mm.isnan(f) or _mm.isinf(f)) else f   # NaN ทำให้ JSON อ่านไม่ถูกในเบราว์เซอร์
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, (int, np.integer)):
        return int(o)
    return o
json.dump(_clean(STATS), open(os.path.join(OUT, "day21_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, allow_nan=False)
_rmax = max([r["dist_m"] for r in REJ], default=0.0)
log(f"คู่ชื่อที่ถูกตัดทิ้งเพราะไกลเกินเพดาน {len(REJ)} รายจาก "
    f"{len({r['site'] for r in REJ})} จุด · ไกลสุด {_rmax:,.1f} ม.")

P1, P2, P3 = draw_hero(), draw_coverage(), draw_roads()

for f in ["day21_two_sources.png", "day21_fields.png", "day21_roads.png",
          "day21_sites_osm.csv", "day21_fields.csv", "day21_edit_queue.csv",
          "day21_stats.json", "day21_osm_named.geojson"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day21_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day21-texts")
assert sum(counts[k] for k in ORDER) == 83
assert named == counts[ORDER[0]] + counts[ORDER[1]]
assert AGG["node_matches"] + AGG["way_matches"] + AGG["polygon_matches"] == int((M.lvl >= 1).sum())
assert 0 <= AGG["dist_min_m"] and AGG["dist_max_m"] <= R_MATCH
assert AGG["road_sites"] + AGG["road_none"] == 83
assert counts[ORDER[5]] == int(((M.n_elem == 0) & ~M.in_protected).sum()), \
    "ระดับ 5 ต้องเท่ากับจุดที่ไม่มีวัตถุในระยะจับคู่และไม่อยู่ในผืนที่ OSM วาด"
assert AGG["sites_in_protected"] >= AGG["polygon_matches"]
# ป้ายสถิติของข้อความจัดอันดับตรวจจริงที่ขั้น check_series (guard_stat_labels) ซึ่ง
# นับเฉพาะบรรทัดที่มีตัวเลข/หน่วย เป็นข้ออ้างการจัดอันดับ ไม่ใช่หัวเรื่องทั่วไป
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps({k: AGG[k] for k in
                          ["named_total", "named_pct", "dist_median_m", "dist_max_m",
                           "road_sites", "road_median_m", "road_none", "edit_queue_rows",
                           "inside_poi_sites", "elem_median", "elem_max"]}, ensure_ascii=False))
print("DONE day21 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()